In [1]:
!pip install pyhealth

1. Data pre-processing and handling

2. Model Architecture

3. Trainning

4. Evaluation

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from typing import Dict, Any

# ==========================================
# 1. MODEL ARCHITECTURE
# ==========================================
class ClinicalTSFTransformer(nn.Module):
    def __init__(self, feature_size=131, d_model=128, nhead=8, num_layers=3):
        super(ClinicalTSFTransformer, self).__init__()

        # We project the 131 raw features to a 'd_model' that IS divisible by nhead
        # 128 / 8 = 16 (Perfect!)
        self.embedding = nn.Linear(feature_size, d_model)

        self.pos_emb = nn.Parameter(torch.zeros(1, 100, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, # Use the projected dimension here
            nhead=nhead,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Heads project back from d_model to original space or classification space
        self.forecasting_head = nn.Linear(d_model, feature_size)
        self.sepsis_head = nn.Linear(d_model, 1)

    def forward(self, x, y=None, **kwargs):
        # x shape: [Batch, Time, 131]

        # Project 131 -> 128
        x_in = self.embedding(x) + self.pos_emb[:, :x.size(1), :]

        # Transformer now sees 128-dim vectors
        h = self.transformer(x_in)

        # Map back to 131 for reconstruction
        recon = self.forecasting_head(h)
        # Map to 1 for sepsis
        logits = self.sepsis_head(h[:, -1, :])

        return {
            "logits": logits,
            "y_prob": torch.sigmoid(logits),
            "reconstruction": recon
        }

# ==========================================
# 2. PREPROCESSING & DATA GENERATOR
# ==========================================
def prepare_fake_clinical_data(n_patients=200, seq_len=24, n_features=131):
    """Generates synthetic eICU-like data for testing the pipeline."""
    # Features: [Batch, Time, Features]
    X = torch.randn(n_patients, seq_len, n_features)
    # Mask: 1 if observed, 0 if missing (eICU is sparse!)
    mask = (torch.rand(n_patients, seq_len, n_features) > 0.2).float()
    X = X * mask
    # Labels: 1 for Sepsis, 0 for Healthy
    y = torch.randint(0, 2, (n_patients,)).float()

    # Split 80/20
    split = int(n_patients * 0.8)
    return (X[:split], y[:split], mask[:split]), (X[split:], y[split:], mask[split:])

# ==========================================
# 3. TRAINING & EVALUATION FUNCTIONS
# ==========================================
def train_model(model, train_data, epochs=10, lr=0.001):
    X, y, mask = train_data
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion_cls = nn.BCEWithLogitsLoss()
    criterion_mse = nn.MSELoss(reduction='none') # 'none' to apply mask manually

    model.train()
    print("Starting Training...")
    for epoch in range(epochs):
        optimizer.zero_grad()

        # Forward
        out = model(X, y=y)

        # Loss 1: Classification (Sepsis)
        loss_cls = criterion_cls(out['logits'].view(-1), y)

        # Loss 2: Masked Forecasting (MSE only on observed values)
        mse_raw = criterion_mse(out['reconstruction'], X)
        loss_recon = (mse_raw * mask).sum() / (mask.sum() + 1e-8)

        # Total Loss (MTL weighting)
        total_loss = loss_cls + (0.5 * loss_recon)

        total_loss.backward()
        optimizer.step()

        if (epoch+1) % 2 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Loss: {total_loss.item():.4f} (Cls: {loss_cls.item():.4f}, Recon: {loss_recon.item():.4f})")

def run_eval(model, test_data, label="TEST"):
    X, y, mask = test_data
    model.eval()
    with torch.no_grad():
        out = model(X)
        probs = out['y_prob'].view(-1)
        preds = (probs > 0.5).float()

        # Metrics
        acc = (preds == y).float().mean()
        tp = ((preds == 1) & (y == 1)).sum().item()
        fn = ((preds == 0) & (y == 1)).sum().item()
        recall = tp / (tp + fn + 1e-8)

        print(f"\n--- {label} RESULTS ---")
        print(f"Accuracy: {acc:.2%} | Recall: {recall:.2%}")
        print(f"Forecasting MSE: {torch.mean((out['reconstruction'] - X)**2).item():.4f}")

# ==========================================
# 4. EXECUTION
# ==========================================
# 1. Setup
train_set, test_set = prepare_fake_clinical_data()
model = ClinicalTSFTransformer(feature_size=131)

# 2. Train
train_model(model, train_set, epochs=10)

# 3. Evaluate
run_eval(model, train_set, label="TRAIN")
run_eval(model, test_set, label="TEST")

Starting Training...
Epoch [2/10] | Loss: 1.6606 (Cls: 1.0334, Recon: 1.2544)
Epoch [4/10] | Loss: 1.2859 (Cls: 0.7079, Recon: 1.1559)
Epoch [6/10] | Loss: 1.1113 (Cls: 0.5880, Recon: 1.0465)
Epoch [8/10] | Loss: 0.9620 (Cls: 0.4524, Recon: 1.0191)
Epoch [10/10] | Loss: 0.8301 (Cls: 0.3219, Recon: 1.0163)

--- TRAIN RESULTS ---
Accuracy: 93.12% | Recall: 89.47%
Forecasting MSE: 0.8144

--- TEST RESULTS ---
Accuracy: 52.50% | Recall: 43.75%
Forecasting MSE: 0.8219


In [80]:
patient_df = pd.read_csv(os.path.join(folder, "patient.csv.gz"))
print(f"Total Unique Patients/Stays in Demo: {patient_df['patientunitstayid'].nunique()}")

Total Unique Patients/Stays in Demo: 2520
